### React
- Reasion(추론)
- Acting(행동)
- 두개를 결합한 LLM 에이전트

- 1.사고
- 2.행동
- 3.관찰
- 4.재추론
을 반복하면서 문제를 해결하도록 만든 구조

### Prompting 한계
- 질문 -> 답변
    - 환각, 최신정보가 부족, 외부시스템 접근 불가, 추론한계
    ### Chain of Thought(Cot)
        - 질문 -단계별 사고-답변
        - 검색/api 호출 불가
        - 정확도 부족
### ReAct 구조
- 루프반복
    - Action
    - Observation
    - Thought
    - Action
    - Observation
    - Thought    
    - Final Answer

    - 생각
    - 도구사용
    - 결과를 관찰
    - 다시생각

- Zero-shot --> Few-shot --> Chain-of_Thought --> ReAct

In [45]:
# 프롬프팅 시뮬레이션
question = '대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?'

# Zero-shot  예시나 질문없이 바로 질문
zero_shot_prompt = f'''질문:{question}\n답변:'''
few_shot_prompt = f'''질문:일본의 수도는 어디이며, 그 도시의 인구는 얼마인가
답변: 일본의 수도는 도쿄이며,인구는 약 1,500만 명입니다.
질문:{question}\n답변:'''
cot_prompt = f'''질문:{question}\n단계별로 생각해보겠습니다.\n1단계: 대한민국의 수도를 확인합니다.
2단계: 해당 도시의 인구 데이터를 조회합니다.\n답변:'''
# 추론과 행동을 교차 반복
react_prompt = f'''질문:{question}
Thought:이 질문에 답하려면 두 가지 정보가 필요합니다. 먼저 대한민국의 수도를 확인하고, 그 다음 해당도시의 인구를 조회해야합니다.
Action : Search[대한민국수도]
Observation: 대한민국의 수도는 서울특별시입니다.
Thought: 서울이 대한민국의 수도인것을 확인했습니다. 이제 서울의인구를 조회해야 합니다.
Action : Search[서울특별시 인구]
Observation:서울특별시의 인구는 약 900만명입니다.
Thought:두 가지 정보를 모두 확인했으므로 최종 답변을 구성합니다
Action: Finish[대한민국의 수도는 서울이며, 서울의 인구는 약 900만 명입니다.]
'''

In [46]:
# React 루프구조 시뮬레이션
class ReActStep:
    '''React의 단일단계(Thought-Action-Observation)을 표현'''
    def __init__(self,step_num, thought, action,action_input,observation):
        self.step_num = step_num 
        self.thought  =thought
        self.action=action
        self.action_input=action_input
        self.observation=observation
    def display(self):
        print(f'-- step {self.step_num} --')
        print(f'Thought {self.thought}')
        print(f'Action {self.action}[{self.action_input}]')
        print(f'Observation {self.observation}')
        print()
class ReActTrace:
    """ReAct의 전체 추론 과정(trace)을 관리하는 클래스"""
    def __init__(self, question):
        self.question = question
        self.steps = []
        self.final_answer = None

    def add_step(self, thought, action, action_input, observation):
        step = ReActStep(len(self.steps) + 1, thought, action, action_input, observation)
        self.steps.append(step)
        return self

    def finish(self, answer):
        self.final_answer = answer
        return self

    def display(self):
        print("=" * 60)
        print(f"Question: {self.question}")
        print("=" * 60)
        for step in self.steps:
            step.display()
        if self.final_answer:
            print(f"Final Answer: {self.final_answer}")
        print("=" * 60)


# 사용 예시: 다단계 질문에 대한 ReAct 트레이스
trace = ReActTrace("2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?")

trace.add_step(
    thought="이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.",
    action="Search",
    action_input="2024년 노벨 물리학상 수상자",
    observation="2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다."
)

trace.add_step(
    thought="수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.",
    action="Search",
    action_input="존 홉필드 제프리 힌턴 연구 분야",
    observation="존 홉필드는 홉필드 네트워크(연상 기억 모델)를, 제프리 힌턴은 볼츠만 머신과 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다."
)

trace.finish(
    "2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, "
    "이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다."
)

trace.display()        

Question: 2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?
-- step 1 --
Thought 이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.
Action Search[2024년 노벨 물리학상 수상자]
Observation 2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다.

-- step 2 --
Thought 수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.
Action Search[존 홉필드 제프리 힌턴 연구 분야]
Observation 존 홉필드는 홉필드 네트워크(연상 기억 모델)를, 제프리 힌턴은 볼츠만 머신과 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다.

Final Answer: 2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, 이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다.


In [47]:
# ReAct vs CoT vs 기타 기법 비교 분석표
# 각 기법의 핵심 특성을 체계적으로 비교합니다.

comparison = {
    "기법": ["Zero-shot", "Few-shot", "CoT", "ReAct"],
    "외부 도구 사용": ["X", "X", "X", "O"],
    "추론 과정 명시": ["X", "X", "O", "O"],
    "다단계 정보 수집": ["X", "X", "X", "O"],
    "사실 검증 가능": ["X", "X", "X", "O"],
    "환각 감소": ["낮음", "중간", "중간", "높음"],
    "복잡한 질문 처리": ["제한적", "제한적", "양호", "우수"]
}

# 포맷팅된 비교표 출력
headers = list(comparison.keys())
print(f"{'기법':<12} {'외부 도구 사용':<16} {'추론 과정 명시':<16} {'다단계 정보 수집':<18} {'사실 검증 가능':<16} {'환각 감소':<12} {'복잡한 질문 처리':<16}")
print("-" * 140)
for i in range(len(comparison["기법"])):
    row = [comparison[key][i] for key in headers]
    print(f"{row[0]:<20} {row[1]:<20} {row[2]:<20} {row[3]:<20} {row[4]:<20} {row[5]:<16} {row[6]}")

기법           외부 도구 사용         추론 과정 명시         다단계 정보 수집          사실 검증 가능         환각 감소        복잡한 질문 처리       
--------------------------------------------------------------------------------------------------------------------------------------------
Zero-shot            X                    X                    X                    X                    낮음               제한적
Few-shot             X                    X                    X                    X                    중간               제한적
CoT                  X                    O                    X                    X                    중간               양호
ReAct                O                    O                    O                    O                    높음               우수


### 1. ReAct 프롬프트 템플릿 설계 원칙
- **시스템 프롬프트의 역할**: 에이전트의 페르소나, 목표, 사용 가능한 도구, 응답 형식 등의 규칙을 정의합니다.
- **도구(Tool) 설명**: 각 도구의 이름, 기능, 입력 형식에 대한 명확한 지시가 포함되어야 합니다.
- **Few-shot 예제**: 모델이 형식에 맞게 출력하도록 가이드 역할을 하는 예제(Thought, Action, Observation 과정 포함)를 제공하는 것이 매우 중요합니다.
- **출력 형식 제약 조건 명시**: Thought, Action, Observation의 정확한 텍스트 패턴과 최종 답변(Finish) 형식을 명시해야 파서가 쉽게 처리할 수 있습니다.

In [48]:
class ReactPromptTemplate:
    def __init__(self):
        self.tools = []
        self.examples = []
        self.system_instruction = ""
    def add_tool(self, name,description,usage_example):
        self.tools.append({'name':name, 'description':description,'usage':usage_example})
        return self
    def add_example(self, question,react_trace):
        self.examples.append({'question':question, 'react_trace':react_trace})
        return self
    def set_system_instruction(self,instruction):
        self.system_instruction = instruction
        return self
    def build(self,user_question):
        prompt_parts = []
        
        # System instruction
        prompt_parts.append(self.system_instruction)
        prompt_parts.append("")
        
        # Tool descriptions
        prompt_parts.append("사용 가능한 도구:")
        for tool in self.tools:            
            prompt_parts.append(f"  - {tool['name']}: {tool['description']}")
            prompt_parts.append(f"    사용법: {tool['usage']}")
        prompt_parts.append("")
        
        # Output format
        prompt_parts.append("출력 형식:")
        prompt_parts.append("Thought: [현재 상황에 대한 추론]")
        prompt_parts.append("Action: [도구 이름][입력값]")
        prompt_parts.append("Observation: [도구 실행 결과]")
        prompt_parts.append("... (필요한 만큼 반복)")
        prompt_parts.append("Thought: [최종 추론]")
        prompt_parts.append("Action: Finish[최종 답변]")
        prompt_parts.append("")
        
        # Few-shot examples
        if self.examples:
            prompt_parts.append("예시:")
            for i, ex in enumerate(self.examples, 1):
                prompt_parts.append(f"--- 예시 {i} ---")
                prompt_parts.append(f"Question: {ex['question']}")
                prompt_parts.append(ex['react_trace'])
                prompt_parts.append("")
        
        # User question
        prompt_parts.append(f"Question: {user_question}")
        
        return "\n".join(prompt_parts)

In [49]:
template = ReactPromptTemplate()
template.set_system_instruction(
    "당신은 주어진 질문에 정학하게 답변하기위해 도구를 사용하는 AI 에이전트입니다\n"
    "반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다"
)
template.add_tool('Search','위키피디아에서 정보를 검색합니다.','Search[검색어]')
template.add_tool('Lookup','현재 문서에서 특정 키워드를 찾습니다.','Lookup[키워드]')
template.add_tool('Calulator','수학 계산을 수행합니다.','Calculate[수식]')

example_trace = """Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]"""

template.add_example("프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?", example_trace)

prompt = template.build("독일의 수도는 어디이며, 면적은 얼마인가?")
print(prompt)


당신은 주어진 질문에 정학하게 답변하기위해 도구를 사용하는 AI 에이전트입니다
반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다

사용 가능한 도구:
  - Search: 위키피디아에서 정보를 검색합니다.
    사용법: Search[검색어]
  - Lookup: 현재 문서에서 특정 키워드를 찾습니다.
    사용법: Lookup[키워드]
  - Calulator: 수학 계산을 수행합니다.
    사용법: Calculate[수식]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

예시:
--- 예시 1 ---
Question: 프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?
Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]

Question: 독일의 수도는 어디이며, 면적은 얼마인가?


### ReAct 출력 파싱
- 정규표현식 기반 : 문자열 패턴 매칭을 통해서 응답에서 Thought, Action, Objservation등의요소를 안정적으로 추출
- 에지케이스 처리 : 특수문자나 잘못된 형식등.. 모델의 예측불가능한 출력을 처리하는 로직이 필요

In [ ]:
import re
from typing import List, Dict ,Optional,Tuple
class ReActParser:
    THOUGHT_PATTERN = re.compile(r'Thought\s*:\s*(.+?)(?=\nAction|$)', re.DOTALL)
    ACTION_PATTERN = re.compile(r'Action\s*:\s*(\w+)\[(.+?)\]', re.DOTALL)
    OBSERVATION_PATTERN = re.compile(r'Observation\s*:\s*(.+?)(?=\nThought|$)', re.DOTALL)
    FINISH_PATTERN = re.compile(r'Action\s*:\s*Finish\[(.+?)\]', re.DOTALL)
